# LeNet-5 실습: 합성곱과 풀링을 직접 계산하기

CNN의 뼈대인 `합성곱 -> 활성화 -> 풀링` 흐름을 작은 이미지로 확인합니다.


In [ ]:
# numpy는 숫자 배열 계산, matplotlib은 이미지 확인에 사용합니다.
import numpy as np
import matplotlib.pyplot as plt

# 실험을 재현할 수 있도록 난수 시드를 고정합니다.
np.random.seed(7)


In [ ]:
# 손글씨 숫자처럼 보이는 8x8 장난감 이미지를 만듭니다.
image = np.array([
    [0,0,1,1,1,1,0,0],
    [0,1,0,0,0,0,1,0],
    [0,0,0,0,0,0,1,0],
    [0,0,0,1,1,1,0,0],
    [0,0,0,0,0,0,1,0],
    [0,1,0,0,0,0,1,0],
    [0,0,1,1,1,1,0,0],
    [0,0,0,0,0,0,0,0],
], dtype=float)

# 작은 노이즈를 더해 실제 손글씨처럼 완벽하지 않은 입력을 만듭니다.
image = np.clip(image + np.random.normal(0, 0.08, image.shape), 0, 1)
plt.imshow(image, cmap="gray")
plt.title("Toy handwritten digit")
plt.axis("off")
plt.show()


In [ ]:
# 2차원 단일 채널 합성곱을 직접 구현합니다.
def conv2d_single_channel(x, kernel):
    # x는 2차원 이미지, kernel은 2차원 필터입니다.
    kh, kw = kernel.shape
    h, w = x.shape
    output = np.zeros((h - kh + 1, w - kw + 1))

    # 필터를 한 칸씩 움직이며 지역 영역과 필터의 곱의 합을 계산합니다.
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            patch = x[i:i+kh, j:j+kw]
            output[i, j] = np.sum(patch * kernel)
    return output

# 세로선과 가로선에 반응하는 간단한 필터입니다.
vertical_edge = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
horizontal_edge = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])
vertical_response = conv2d_single_channel(image, vertical_edge)
horizontal_response = conv2d_single_channel(image, horizontal_edge)

# 필터가 어떤 위치에서 강하게 반응하는지 시각화합니다.
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, data, title in zip(axes, [image, vertical_response, horizontal_response], ["input", "vertical edge", "horizontal edge"]):
    ax.imshow(data, cmap="coolwarm" if "edge" in title else "gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()


In [ ]:
# max pooling은 작은 영역에서 가장 강한 특징만 남깁니다.
def max_pool2d(x, pool_size=2, stride=2):
    # 출력 크기를 계산하고 빈 배열을 준비합니다.
    h, w = x.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))

    # pool_size x pool_size 영역에서 최댓값만 저장합니다.
    for i in range(out_h):
        for j in range(out_w):
            si, sj = i * stride, j * stride
            output[i, j] = np.max(x[si:si+pool_size, sj:sj+pool_size])
    return output

# 합성곱 결과에 ReLU를 적용한 뒤 풀링합니다.
relu_response = np.maximum(vertical_response, 0)
pooled = max_pool2d(relu_response)
print("입력 shape:", image.shape)
print("합성곱 후 shape:", vertical_response.shape)
print("풀링 후 shape:", pooled.shape)
plt.imshow(pooled, cmap="gray")
plt.title("pooled feature map")
plt.axis("off")
plt.show()


## 관찰 포인트

합성곱은 같은 필터를 모든 위치에 공유하고, 풀링은 작은 이동에 둔감한 표현을 만듭니다.
